md# 01 · Station graph and edge-signal test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sladekd/fog-obs-fusion/blob/main/notebooks/01_station_graph.ipynb)

Builds a Delaunay graph over the SYNOP stations, prunes long edges, checks connectivity, and tests whether graph edges share more signal (visibility anomalies, fog co-occurrence) than non-edge pairs **at the same distance**.

In [ ]:
# Install the package from GitHub (Colab). Locally: pip install -e .
!pip install -q git+https://github.com/Sladekd/fog-obs-fusion.git

In [ ]:
import pandas as pd

# Load your decoded SYNOP table (columns WMO_ID, ANO, MES, DIA, HORA, visibility_m, ...).
# In Colab, e.g. mount Drive first:
#   from google.colab import drive; drive.mount("/content/drive")
final_df = pd.read_parquet("/content/drive/MyDrive/synop/final_df.parquet")   # <- edit path
final_df.head()

md## Station metadata and graph

In [ ]:
from fogfusion.io import load_stations, check_coverage
from fogfusion.graph import build_delaunay_graph

stations = check_coverage(final_df, load_stations())   # bundled Czech station table
sg = build_delaunay_graph(stations)                     # cutoff: 95th pct of edge length
print(sg.summary())

In [ ]:
import matplotlib.pyplot as plt
from fogfusion.viz import plot_station_graph

fig, ax = plt.subplots(figsize=(11, 7))
plot_station_graph(sg, ax=ax)
plt.tight_layout(); plt.show()

sg.edges[sg.edges.kept].nlargest(5, "abs_delev_m")[["u_name", "v_name", "dist_km", "abs_delev_m"]]

md## Do edges carry signal beyond distance?

Positive differences mean Delaunay neighbours share more than equally distant non-neighbours. A negative elevation effect supports elevation-weighted edges.

In [ ]:
from fogfusion.io import anomalies, fog_matrix
from fogfusion.graph import pair_stats, distance_matched, elevation_effect

anom = anomalies(final_df, "visibility_m", log=True)
fog = fog_matrix(final_df)
pairs = pair_stats(anom, fog, sg.stations, sg.graph)

for col in ["corr", "fog_cooc"]:
    diff, bins = distance_matched(pairs, col)
    print(f"{col:9s} edge - non-edge: {diff:+.3f} over {len(bins)} distance bins")
print(f"Spearman(|elevation difference|, corr) along edges: {elevation_effect(pairs):+.2f}")

In [ ]:
from fogfusion.viz import plot_pair_signal
plot_pair_signal(pairs); plt.show()

pairs[pairs.is_edge].nsmallest(5, "corr")[["u_name", "v_name", "dist_km", "abs_delev_m", "corr"]]

md## Arrays for a GNN (Phase 2)

In [ ]:
from fogfusion.gnn import graph_arrays
arr = graph_arrays(sg)
{k: v.shape for k, v in arr.items()}